# NB21 The fee market — Chapter 21

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB21-1: EIP-1559 base fee update, with the integer arithmetic of the reference implementation as is. Checked against the first block of the London fork, then run on a congestion scenario (for study)
BASE_FEE_MAX_CHANGE_DENOMINATOR = 8      # constant in the EIP-1559 text
ELASTICITY_MULTIPLIER = 2
INITIAL_BASE_FEE = 1_000_000_000          # 1 gwei (in wei)

def next_base_fee(parent_base_fee: int, parent_gas_used: int, parent_gas_limit: int) -> int:
    """A function of the parent block only (Roughgarden's idea (2)). Every division is integer division."""
    target = parent_gas_limit // ELASTICITY_MULTIPLIER
    if parent_gas_used == target:
        return parent_base_fee
    if parent_gas_used > target:
        delta = parent_base_fee * (parent_gas_used - target) // target // BASE_FEE_MAX_CHANGE_DENOMINATOR
        return parent_base_fee + max(delta, 1)
    delta = parent_base_fee * (target - parent_gas_used) // target // BASE_FEE_MAX_CHANGE_DENOMINATOR
    return parent_base_fee - delta

# Check 1: first block of the London fork, 12,965,000 (2021-08-05T12:33:42Z) → 12,965,001. Header values are the raw responses of two public RPC endpoints (see the footnote)
b_12965000, used, limit = INITIAL_BASE_FEE, 30_025_257, 30_029_122
calc = next_base_fee(b_12965000, used, limit)
print(f"Block 12,965,000: base fee {b_12965000:,} wei, gas {used:,}/{limit:,} (target {limit // 2:,})")
print(f"Computed base fee of 12,965,001: {calc:,} wei; matches the value on the chain, 1,124,967,822: {calc == 1_124_967_822}")

# Check 2: a full block gives exactly +12.5%, an empty block -12.5%
print(f"After a full block: {next_base_fee(10**9, 60_000_000, 60_000_000) / 10**9:.4f} gwei, after an empty block: {next_base_fee(10**9, 0, 60_000_000) / 10**9:.4f} gwei")

# Scenario: at the gas limit observed on the reference date, 60,000,000 (target 30,000,000), congestion (20 full blocks) → easing (20 blocks at 1/4 of the target) → normal (at the target)
limit = 60_000_000                        # observed 2026-10-24 (not a constant; Appendix A-10)
scenario = [limit] * 20 + [limit // 8] * 20 + [limit // 2] * 5
b = 10**9
print(f"\n{'Block':>4} {'Gas used':>12} {'Base fee (gwei)':>18} {'Change':>8}")
for i, g in enumerate(scenario, 1):
    nb = next_base_fee(b, g, limit)
    if i <= 3 or i in (10, 20, 21, 22, 23, 30, 40, 41, 45):
        print(f"{i:>4} {g:>12,} {nb / 10**9:>18.4f} {(nb / b - 1):>+8.2%}")
    b = nb
print("After 20 full blocks in a row the base fee is (1.125)^20 ≈ {:.1f} times as high. When demand falls it comes down (by 12.5% per block if the blocks are empty).".format(1.125 ** 20))

Block 12,965,000: base fee 1,000,000,000 wei, gas 30,025,257/30,029,122 (target 15,014,561)
Computed base fee of 12,965,001: 1,124,967,822 wei; matches the value on the chain, 1,124,967,822: True
After a full block: 1.1250 gwei, after an empty block: 0.8750 gwei

Block     Gas used    Base fee (gwei)   Change
   1   60,000,000             1.1250  +12.50%
   2   60,000,000             1.2656  +12.50%
   3   60,000,000             1.4238  +12.50%
  10   60,000,000             3.2473  +12.50%
  20   60,000,000            10.5451  +12.50%
  21    7,500,000             9.5565   -9.37%
  22    7,500,000             8.6606   -9.37%
  23    7,500,000             7.8486   -9.37%
  30    7,500,000             3.9403   -9.37%
  40    7,500,000             1.4723   -9.37%
  41   30,000,000             1.4723   +0.00%
  45   30,000,000             1.4723   +0.00%
After 20 full blocks in a row the base fee is (1.125)^20 ≈ 10.5 times as high. When demand falls it comes down (by 12.5% per block if the

In [ ]:
# NB21-2: first-price auction vs 1559, with the same users and the same block space. How the guess "what will others bid?" enters what is paid (for study)
# Valuation v is willingness to pay per unit of gas (a unitless model value). Block capacity C = 4 transactions, all of the same size 1. Miner's marginal cost μ = 0.
users = {"A": 10.0, "B": 8.0, "C": 7.5, "D": 7.0, "E": 6.0, "F": 3.0}      # valuations (true)
C = 4

def fpa(bids):
    """First-price auction: include the top C bids; each pays its own bid, all of it to the miner."""
    order = sorted(bids.items(), key=lambda kv: -kv[1])[:C]
    return {u: b for u, b in order}, sum(b for _, b in order), 0.0

def mech1559(valuations, base_fee, tip=0.01):
    """1559: cap = valuation, tip = a small value near μ (the 'obvious optimal bid' of Theorem 5.7). The base fee is burned; only the tip goes to the miner."""
    eligible = sorted(((u, v) for u, v in valuations.items() if v >= base_fee), key=lambda kv: -kv[1])[:C]
    paid = {u: min(base_fee + tip, v) for u, v in eligible}
    burned = base_fee * len(eligible)
    return paid, sum(paid.values()) - burned, burned

print("(a) FPA: if users bid their true valuations as they are, they overpay")
paid, miner, burned = fpa(users)
print(f"   Included: {paid}  miner revenue {miner:.2f}, burned 0")
print("   A paid 10 but would have got in by paying just over 6 (highest losing bid E = 6), so in an FPA bidding your valuation is not a dominant strategy (not DSIC)")

print("\n(a′) FPA: imitating something like an equilibrium in which everyone shades down to 'near the highest losing bid'")
shaded = {u: round(v * 0.8, 2) for u, v in users.items()}
paid, miner, burned = fpa(shaded)
print(f"   Included: {paid}  miner revenue {miner:.2f}; how far to shade depends on guesses about others' bids")

for base_fee in [6.5, 7.2, 3.0]:
    paid, miner, burned = mech1559(users, base_fee)
    label = "adequate" if base_fee == 6.5 else ("high" if base_fee == 7.2 else "excessively low")
    print(f"\n(b) 1559, base fee {base_fee} ({label}): included {paid}")
    n_elig = sum(1 for v in users.values() if v >= base_fee)
    if n_elig > C:
        print(f"   Burned {burned:.2f}, miner revenue (tips) {miner:.2f}. Eligible users {n_elig} > capacity {C}: this code does not compute the tip competition (the exception in Theorem 5.7; see the next line).")
    else:
        print(f"   Burned {burned:.2f}, miner revenue (tips) {miner:.2f}. Users need not guess others' bids (cap = valuation).")
print("\nWhen the base fee is excessively low, like 3.0, more users are willing to pay it than there is capacity, and tip competition comes back: this is where 1559 reverts to an FPA (the conditions of Theorem 5.7).")

(a) FPA: if users bid their true valuations as they are, they overpay
   Included: {'A': 10.0, 'B': 8.0, 'C': 7.5, 'D': 7.0}  miner revenue 32.50, burned 0
   A paid 10 but would have got in by paying just over 6 (highest losing bid E = 6), so in an FPA bidding your valuation is not a dominant strategy (not DSIC)

(a′) FPA: imitating something like an equilibrium in which everyone shades down to 'near the highest losing bid'
   Included: {'A': 8.0, 'B': 6.4, 'C': 6.0, 'D': 5.6}  miner revenue 26.00; how far to shade depends on guesses about others' bids

(b) 1559, base fee 6.5 (adequate): included {'A': 6.51, 'B': 6.51, 'C': 6.51, 'D': 6.51}
   Burned 26.00, miner revenue (tips) 0.04. Users need not guess others' bids (cap = valuation).

(b) 1559, base fee 7.2 (high): included {'A': 7.21, 'B': 7.21, 'C': 7.21}
   Burned 21.60, miner revenue (tips) 0.03. Users need not guess others' bids (cap = valuation).

(b) 1559, base fee 3.0 (excessively low): included {'A': 3.01, 'B': 3.01, 'C': 3

In [ ]:
# NB21-3: EIP-4844 base fee per blob gas, porting the text's fake_exponential and constants as they are (for study)
MAX_BLOB_GAS_PER_BLOCK = 786_432          # 6 blobs
TARGET_BLOB_GAS_PER_BLOCK = 393_216       # 3 blobs
MIN_BASE_FEE_PER_BLOB_GAS = 1             # wei
BLOB_BASE_FEE_UPDATE_FRACTION = 3_338_477
GAS_PER_BLOB = 2**17

def fake_exponential(factor: int, numerator: int, denominator: int) -> int:
    """EIP-4844 Helpers: integer approximation of factor * e**(numerator/denominator) by Taylor expansion (as in the text)."""
    i = 1
    output = 0
    numerator_accum = factor * denominator
    while numerator_accum > 0:
        output += numerator_accum
        numerator_accum = (numerator_accum * numerator) // (denominator * i)
        i += 1
    return output // denominator

def calc_excess_blob_gas(parent_excess: int, parent_blob_gas_used: int) -> int:
    if parent_excess + parent_blob_gas_used < TARGET_BLOB_GAS_PER_BLOCK:
        return 0
    return parent_excess + parent_blob_gas_used - TARGET_BLOB_GAS_PER_BLOCK

def base_fee_per_blob_gas(excess: int) -> int:
    return fake_exponential(MIN_BASE_FEE_PER_BLOB_GAS, excess, BLOB_BASE_FEE_UPDATE_FRACTION)

import math
print(f"Rate of change per block filled to the maximum (6 blobs; the target is 3): e^(393216/3338477) = {math.exp(TARGET_BLOB_GAS_PER_BLOCK / BLOB_BASE_FEE_UPDATE_FRACTION):.4f}")
# Scenario: 30 blocks in a row with 6 blobs (the maximum) → 30 blocks with 0 blobs → the target (3 blobs)
excess = 0
print(f"{'Block':>4} {'Blobs':>6} {'excess_blob_gas':>16} {'Base fee per blob gas (wei)':>26}")
for i, blobs in enumerate([6] * 30 + [0] * 30 + [3] * 3, 1):
    fee = base_fee_per_blob_gas(excess)
    if i in (1, 2, 3, 10, 20, 30, 31, 32, 40, 50, 60, 61, 63):
        print(f"{i:>4} {blobs:>6} {excess:>16,} {fee:>26,}")
    excess = calc_excess_blob_gas(excess, blobs * GAS_PER_BLOB)
print("After 30 maximum blocks the excess is 30 × 393,216 = {:,} and the fee is e^(that value/3,338,477) ≈ {:.0f} times as high. After 30 empty blocks the excess returns to 0 and the fee to the minimum of 1 wei.".format(30 * 393_216, math.exp(30 * 393_216 / 3_338_477)))

Rate of change per block filled to the maximum (6 blobs; the target is 3): e^(393216/3338477) = 1.1250
Block  Blobs  excess_blob_gas Base fee per blob gas (wei)
   1      6                0                          1
   2      6          393,216                          1
   3      6          786,432                          1
  10      6        3,538,944                          2
  20      6        7,471,104                          9
  30      6       11,403,264                         30
  31      0       11,796,480                         34
  32      0       11,403,264                         30
  40      0        8,257,536                         11
  50      0        4,325,376                          3
  60      0          393,216                          1
  61      3                0                          1
  63      3                0                          1
After 30 maximum blocks the excess is 30 × 393,216 = 11,796,480 and the fee is e^(that value/3,338,477) ≈ 34 ti